Inputs to the analyis: `image.tif`, `pca.tif`, `msu_pts.gpkg`
* Read in all inputs.
* Identify smaller AOI tile.
* Extract DINOv3 embedding for the window that contains the MSU tree point.
* Extract PCA vector from pca.tif at the MSU tree point location.
* Create table with x, y, species, token_row, token_col, pca_vec, embed_vec
* Create two pairs of sets: within species, across species
* Compute cosine distance for some pairs
* Create visualization to show results

test site: `kev_2023-07-21`

In [3]:
import sys
import os
from pathlib import Path
sys.path.insert(0, str(Path("../dinov3").resolve()))
sys.path.insert(0, str(Path('../src').resolve()))

import cosine_similarity as cs

%load_ext autoreload
%autoreload 2

## Identify smaller AOI tile

In [12]:
cs.aoi_clip(
    image_path="../data/msu_images/kev_2023-07-21_cropped.tif",
    pca_path="../data/pca/kev_2023-07-21_cropped_pca.tif",
    trees_path="../data/msu_field/_clean/kev_fielddata_clean.gpkg",
    img_out_path="../data/tmp/kev_2023-07-21_maxar_aoi.tif",
    pca_out_path="../data/tmp/kev_2023-07-21_pca_aoi.tif",
    buffer_m=30.0,
)

Tree CRS: EPSG:3857  | Image CRS: EPSG:32737
Wrote AOI clip for ../data/msu_images/kev_2023-07-21_cropped.tif to ../data/tmp/kev_2023-07-21_maxar_aoi.tif
Wrote AOI clip for ../data/pca/kev_2023-07-21_cropped_pca.tif to ../data/tmp/kev_2023-07-21_pca_aoi.tif


## Extract features for each tree location

In [10]:
results = cs.per_tree_features("../data/tmp/kev_2023-07-21_maxar_aoi.tif",
                              "../data/tmp/kev_2023-07-21_pca_aoi.tif",
                              "../data/msu_field/_clean/kev_fielddata_clean.gpkg",)

In [9]:
results[0]

{'x': 4081855.9763168166,
 'y': -85893.62297953294,
 'species': 'cedar',
 'r_tok': 47,
 'c_tok': 47,
 'pca_vec': array([], shape=(3, 0, 0), dtype=float32),
 'embed_vec': array([ 1.00158639e-01,  2.92862236e-01,  1.18375158e+00, -8.13715696e-01,
        -5.21809399e-01,  9.27432477e-01, -1.17553759e+00,  1.94554675e+00,
         7.87294090e-01,  1.49317622e+00,  8.67643178e-01, -1.37802303e+00,
        -1.84366977e+00,  1.14001238e+00, -3.15251261e-01, -7.49729812e-01,
         8.95124018e-01, -1.91817448e-01,  2.30261222e-01,  6.95192933e-01,
         3.39314431e-01, -2.05506943e-02, -6.60057783e-01,  1.07779002e+00,
         1.29222572e+00,  6.16071105e-01, -4.01648581e-01,  7.52611637e-01,
        -1.48334074e+00, -6.61420166e-01,  5.52446127e-01, -6.41597092e-01,
        -7.78568506e-01, -5.17315984e-01,  3.11757743e-01, -5.06881177e-01,
        -5.65497696e-01, -1.97435832e+00, -1.14370024e+00,  3.44353884e-01,
         1.56182492e+00,  3.34097743e-01,  1.78931028e-01,  4.41349559e